Load jsons files

In [3]:
from pathlib import Path
import pandas as pd

#jsonl_folder = Path(r"C:\GH-ORDI\DATA\Data-Model-Observability\Anciens\2026-08-06-temp-(00, 1.5)-seed-null")
#jsonl_folder = Path(r"C:\GH-ORDI\DATA\Data-Model-Observability\Anciens\2026-08-06-temp-(00, 1.5)-seed-null-new-client")
#jsonl_folder = Path(r"C:\GH-ORDI\DATA\Data-Model-Observability\Anciens\2026-09-06-temp-(0.7)-top_k-(1-80)-seed-null")
jsonl_folder = Path(r"C:\GH-ORDI\DATA\Data-Model-Observability")

jsonl_df : dict[str, pd.DataFrame] = {}
for file in jsonl_folder.iterdir():

    if file.is_file():
        print(file)
        jsonl_df[file.name] = pd.read_json(file, lines=True)


C:\GH-ORDI\DATA\Data-Model-Observability\gemma3_1b-2026-09-06_14-29-58-x30.jsonl
C:\GH-ORDI\DATA\Data-Model-Observability\gemma3_1b-2026-09-06_14-31-45-x30.jsonl
C:\GH-ORDI\DATA\Data-Model-Observability\gemma3_1b-2026-09-06_14-33-05-x30.jsonl
C:\GH-ORDI\DATA\Data-Model-Observability\gemma3_1b-2026-09-06_14-34-25-x30.jsonl
C:\GH-ORDI\DATA\Data-Model-Observability\gemma3_1b-2026-09-06_14-35-43-x30.jsonl
C:\GH-ORDI\DATA\Data-Model-Observability\gemma3_1b-2026-09-06_14-37-00-x30.jsonl
C:\GH-ORDI\DATA\Data-Model-Observability\gemma3_1b-2026-09-06_14-38-17-x30.jsonl
C:\GH-ORDI\DATA\Data-Model-Observability\gemma3_1b-2026-09-06_14-39-32-x30.jsonl


Extract responses from folder jsonl files into a list of respnse dict per file

In [4]:
import json

response_contents : dict[str, list[dict[str, str]]] = {} # [file_name, list json response parsed into a dict]

# get the responses
nbr_json_error = 0
for jsonl_file in jsonl_df.keys():
    file_responses : list[dict[str, str]] = []

    # process the file responses
    for response in jsonl_df[jsonl_file]['chat_response']:

        response_content_str = response['content']

        try:
            response_content_str = response_content_str.replace("\n", "").replace("```json", "").replace("```", "")
            response_content = json.loads(response_content_str)
            file_responses.append(response_content)
        except:
            #print("bug=", response_content_str)
            nbr_json_error += 1
            continue 

    response_contents[jsonl_file] = file_responses

# =============================== #
print_result = True

if print_result:
    print("nbr_json_error=", nbr_json_error)
    print(len(response_contents))

    for key, value in response_contents.items():
        print(key, len(value))


nbr_json_error= 49
8
gemma3_1b-2026-09-06_14-29-58-x30.jsonl 16
gemma3_1b-2026-09-06_14-31-45-x30.jsonl 30
gemma3_1b-2026-09-06_14-33-05-x30.jsonl 30
gemma3_1b-2026-09-06_14-34-25-x30.jsonl 25
gemma3_1b-2026-09-06_14-35-43-x30.jsonl 22
gemma3_1b-2026-09-06_14-37-00-x30.jsonl 14
gemma3_1b-2026-09-06_14-38-17-x30.jsonl 20
gemma3_1b-2026-09-06_14-39-32-x30.jsonl 20


Calculate the avg nbr of keys found in each file

In [5]:
from statistics import mean, stdev

file_names = list(response_contents.keys())

avg_keys : list[float] = []
std_keys : list[float] = []

for file_name in file_names:

    contents = response_contents[file_name]

    response_nbr_keys = list[int]

    nbr_keys : list[int] = []
    for content in contents:
        data = pd.json_normalize(content)
        nbr_keys.append(len(data.keys()))

    print(file_name, nbr_keys)
    avg_keys.append(mean(nbr_keys))
    std_keys.append(stdev(nbr_keys))

print(avg_keys)
print(std_keys)

gemma3_1b-2026-09-06_14-29-58-x30.jsonl [25, 27, 23, 26, 20, 20, 26, 20, 20, 26, 20, 20, 26, 20, 20, 26]
gemma3_1b-2026-09-06_14-31-45-x30.jsonl [27, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24]
gemma3_1b-2026-09-06_14-33-05-x30.jsonl [23, 23, 26, 26, 27, 26, 24, 24, 26, 23, 23, 28, 22, 22, 23, 23, 24, 27, 21, 24, 20, 23, 25, 23, 23, 22, 23, 23, 24, 22]
gemma3_1b-2026-09-06_14-34-25-x30.jsonl [23, 28, 22, 27, 24, 26, 28, 27, 22, 14, 23, 10, 28, 19, 15, 27, 22, 13, 23, 28, 28, 22, 28, 23, 25]
gemma3_1b-2026-09-06_14-35-43-x30.jsonl [25, 19, 25, 25, 26, 24, 12, 24, 22, 23, 27, 20, 13, 15, 18, 24, 26, 17, 29, 28, 18, 22]
gemma3_1b-2026-09-06_14-37-00-x30.jsonl [26, 19, 21, 17, 16, 26, 26, 23, 25, 26, 15, 32, 29, 12]
gemma3_1b-2026-09-06_14-38-17-x30.jsonl [14, 15, 26, 13, 1, 14, 26, 22, 17, 25, 26, 17, 16, 31, 29, 19, 10, 24, 25, 30]
gemma3_1b-2026-09-06_14-39-32-x30.jsonl [25, 28, 12, 11, 17, 23, 22, 14, 24, 11, 36, 2

Display the avg and stddev of keys found

In [ ]:
import model_observability.core.plot_utils as plt_utils


plt_utils.plot_bar_avg_std("test", avg_keys)